# 9. Detection capability

Can we detect a companion 10⁴ times fainter than its star, 6 λ/D away, behind a vortex coronagraph? This
notebook answers with the tools of a detection study (use case UC7):

1. a **matched filter** (the Neyman-Pearson test of a known signal) applied to many noisy exposures, with and
   without the companion;
2. the empirical **ROC curve**, compared with its analytic form $P_D = Q(Q^{-1}(P_{FA}) - d)$;
3. the **contrast curve** of a residual image (5σ, with the small-sample penalty of Mawet et al. 2014);
4. the **detection capability** versus separation and contrast.

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import lumos.io
from lumos.elements.detectors import Camera
from lumos.fields import DataProduct
from lumos.processing import detection, photometry

## The instrument

The preset `presets/coronagraph.lumos` places a star and a companion of contrast 10⁻⁴ at 6 λ/D behind a
charge-2 vortex. We add a camera with read noise; at λ = 1.6 µm, D = 8 m and f = 100 m, λf/D = 20 µm, i.e.
4 pixels of 5 µm.

In [ ]:
ctx = lumos.io.load("presets/coronagraph.lumos")
camera = ctx.add(
    Camera(
        name="camera",
        exposure=0.3 * u.s,
        read_noise=10 * u.electron,
        bias=1000 * u.adu,
        full_well=1e9 * u.electron,
        bits=32,
    )
)
ctx.connect(ctx["image"].port("out"), camera.port("in"))
star, planet = ctx["star"], ctx["planet"]
contrast = float(planet.flux_density / star.flux_density)
resolution = 20 * u.um  # λf/D


def noiseless(star_on=True, planet_on=True):
    """Expected image in adu, without bias."""
    fluxes = star.flux_density, planet.flux_density
    star.flux_density = fluxes[0] if star_on else 0 * u.Jy
    planet.flux_density = fluxes[1] if planet_on else 0 * u.Jy
    camera.noise = False
    image = ctx.run(["camera.out"])["camera.out"]
    camera.noise = True
    star.flux_density, planet.flux_density = fluxes
    return DataProduct(image.data - camera.bias, axes={k: v.to(u.um) for k, v in image.axes.items()})


reference = noiseless(planet_on=False)  # the star alone: the PSF subtracted from the data
template = noiseless(star_on=False)  # the companion alone: the signal searched for
sigma = np.sqrt(reference.data.value + 10.0**2)  # photon and read noise of one pixel, in adu (gain 1)
print(f"companion: {template.data.value.sum():.0f} e- in total, peak {template.data.value.max():.1f} e-")

## Matched filter and ROC

Each exposure gives a new noise realisation. The residual (frame - bias - star) is filtered by the template
of the companion; without it, the statistic follows N(0, 1), with it N(d, 1), the deflection being
$d = \sqrt{\sum s_i^2/\sigma_i^2}$.

In [ ]:
def residuals(n, with_planet):
    flux = planet.flux_density
    planet.flux_density = flux if with_planet else 0 * u.Jy
    out = []
    for _ in range(n):
        camera.expose()
        frame = ctx.run(["camera.out"])["camera.out"].data.value
        out.append(frame - 1000 - reference.data.value)
    planet.flux_density = flux
    return np.array(out)


h0 = detection.matched_filter(
    residuals(400, False).reshape(400, -1), template.data.value.ravel(), sigma.ravel()
)
h1 = detection.matched_filter(
    residuals(400, True).reshape(400, -1), template.data.value.ravel(), sigma.ravel()
)
d = float(np.sqrt(np.sum(template.data.value**2 / sigma**2)))
print(f"deflection d = {d:.2f}; measured mean without/with companion: {h0.mean():.2f} / {h1.mean():.2f}")

roc = detection.roc_curve(h0, h1)
pfa_model = np.logspace(-4, 0, 200)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(roc.axes["false_alarm_probability"].value, roc.data.value, label="400 + 400 exposures")
ax.plot(
    pfa_model, detection.roc_matched_filter(pfa_model, deflection=d), "--", label=f"analytic, d = {d:.2f}"
)
ax.set(
    xscale="log",
    xlim=(1e-3, 1),
    xlabel="false-alarm probability",
    ylabel="detection probability",
    title="ROC of the matched filter",
)
ax.legend()
fig.tight_layout()

## Contrast curve

Without knowing where the companion is, the detection limit at each separation follows from the noise in an
annulus of width λ/D of a residual image (Mawet et al. 2014). The contrast is relative to the peak of the
off-axis PSF of the star, here the companion template scaled by its contrast.

In [ ]:
residual = DataProduct(residuals(1, False)[0] * u.adu, axes=reference.axes)
peak = template.data.max() / contrast
curve = photometry.contrast_curve(residual, peak=peak, resolution=resolution, sigma=5)
separations = curve.axes["separation"] / resolution
for s, c in zip(separations, curve.data.value, strict=True):
    print(f"{s:4.1f} λ/D: {c:.1e}")

## Detection capability

Moving the companion over separations gives the deflection per unit contrast $k(r)$; the matched filter then
detects a contrast $c$ with probability $Q(Q^{-1}(P_{FA}) - c\,k(r))$.

In [ ]:
theta = planet.theta_x
per_lambda_d = theta / 6
seps = np.arange(2, 10)
k = []
for s in seps:
    planet.theta_x = s * per_lambda_d
    shape = noiseless(star_on=False).data.value
    k.append(np.sqrt(np.sum(shape**2 / sigma**2)) / contrast)
planet.theta_x = theta

contrasts = np.logspace(-6, -2, 81)
capability = detection.detection_capability(seps * resolution, contrasts, deflection_per_contrast=k, pfa=1e-3)
limit = detection.contrast_limit(seps * resolution, deflection_per_contrast=k, pfa=1e-3, pd=0.5)

result = ctx.run(["camera.out"], archive="detection-capability")
fig, ax = plt.subplots(figsize=(6, 4))
mesh = ax.pcolormesh(seps, contrasts, capability.data.value, shading="nearest", cmap="viridis")
ax.plot(seps, limit.data.value, "w--", label="P_D = 50 % (matched filter)")
ax.plot(separations, curve.data.value, "r.-", label="5σ contrast curve (blind search)")
ax.set(yscale="log", xlabel="separation [λ/D]", ylabel="contrast", title="Detection probability, P_FA = 1e-3")
fig.colorbar(mesh, ax=ax, label="P_D")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
result.archive.save_figure(
    fig,
    "detection_capability",
    data={"separation": seps, "contrast": contrasts, "probability": capability.data.value},
)

The matched filter, which knows where the companion is, reaches fainter contrasts than the blind 5σ
search. For the companion of the preset (10⁻⁴ at 6 λ/D), in a single exposure of 0.3 s:

In [ ]:
pd = detection.roc_matched_filter(1e-3, deflection=d)
print(f"matched filter: P_D = {pd:.0%} at P_FA = 1e-3; blind 5σ limit at 6 λ/D: {curve.data.value[5]:.1e}")
n_needed = ((3.09 + 1.28) / d) ** 2  # deflection grows as the square root of the number of exposures
print(f"exposures needed for P_D = 90 %: {np.ceil(n_needed):.0f}")